# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
"""My lane is a "which pages first?" ranking with an observed label (did impressions drop 20%+ in the next 30 days), so from the toolkit: a classifier's probability, judged at precision@K, not its yes/no.
I start with logistic regression. Its readable (one weight per feature, I can explain any ranking to an editor) and my features are already logged and flagged, so a linear model gets a fair shot. Then I check if more complexity earns its keep: a depth-3 decision tree (small enough to print), a random forest (what the reference pipeline picked) and gradient boosting.
My rule for picking: a more complex model only wins if it clearly beats logistic regression at P@50 (my w02 metric) AND doesnt lose at P@20. Same features as w03 (lane_features.py), seeds fixed at 42."""

import sys
sys.path.insert(0, "../scripts")
sys.path.insert(0, "../../scripts")
import numpy as np
import pandas as pd
import sklearn
import lane_features as lf
import lane_baseline as lb
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

raw = lf.load_raw()
df = lf.build_features(raw)
X = lf.feature_matrix(df)
y = df["is_declining_label"].to_numpy()
print(f"{len(df):,} pages | {X.shape[1]} feature columns ({len(lf.NUMERIC_FEATURES)} numeric + one-hot of {', '.join(lf.CATEGORICAL_FEATURES)}) | base rate {y.mean():.3f}")
print(f"pandas {pd.__version__} | numpy {np.__version__} | scikit-learn {sklearn.__version__} | seed {lf.SEED}")

30,000 pages | 28 feature columns (16 numeric + one-hot of competition_level, content_type, main_intent) | base rate 0.542
pandas 2.2.3 | numpy 2.1.3 | scikit-learn 1.6.1 | seed 42


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
"""Grouped by client: 20% of the clients (seed 42) are held out completely. Same split as the reference pipeline and my w04 baseline, so the numbers line up.
Why this is the honest split for my question: the queue is meant to work on pages the model never saw, and pages from one client share a site, a niche and tracking quirks. A random row split lets the model learn the client and look smarter than it is (w06 measures by how much).
Time-aware isnt possible, the file is one snapshot with one label window. The time part is handled inside the features instead: everything is from before day -30."""

train, test = lf.client_holdout(df)
overlap = set(df.loc[train, "client_id"]) & set(df.loc[test, "client_id"])
print(f"train: {train.sum():,} rows from {df.loc[train, 'client_id'].nunique()} clients | test: {test.sum():,} rows from {df.loc[test, 'client_id'].nunique()} clients | clients on both sides: {len(overlap)}")
print(f"base rate train {y[train].mean():.3f} vs test {y[test].mean():.3f}")

test_clients = df[test].groupby("client_id")["is_declining_label"].agg(rows="size", base_rate="mean").sort_values("rows", ascending=False)
test_clients.index = [f"test client {i}" for i in range(1, len(test_clients) + 1)]
display(test_clients.round(3))
print(f"careful: the two biggest test clients are {test_clients['rows'].iloc[:2].sum() / test.sum():.0%} of the test rows, so this one split is noisy. w06 reruns it as 5-fold grouped CV.")

train: 27,675 rows from 26 clients | test: 2,325 rows from 6 clients | clients on both sides: 0
base rate train 0.555 vs test 0.391


,rows,base_rate
test client 1,1106,0.220
test client 2,1031,0.542
test client 3,118,0.619
test client 4,35,0.486
test client 5,32,0.531
test client 6,3,0.000


careful: the two biggest test clients are 92% of the test rows, so this one split is noisy. w06 reruns it as 5-fold grouped CV.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
"""Same data, same split, same metric as the w04 baseline, and the baseline is recomputed right here from the frozen lane_baseline.py (not loaded from an old file)."""

def evaluate(scores):
    s = np.asarray(scores, dtype=float)
    return {"P@20": lf.precision_at_k(y[test], s, 20), "P@50": lf.precision_at_k(y[test], s, 50),
            "P@100": lf.precision_at_k(y[test], s, 100), "P@200": lf.precision_at_k(y[test], s, 200),
            "AUC": roc_auc_score(y[test], s), "avg precision": average_precision_score(y[test], s)}

models = {
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "decision tree (depth 3)": DecisionTreeClassifier(max_depth=3, min_samples_leaf=200, random_state=lf.SEED),
    "random forest": RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=25, n_jobs=-1, random_state=lf.SEED),
    "gradient boosting": HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=200, random_state=lf.SEED),
}
scores = {
    "dummy (random, stratified)": DummyClassifier(strategy="stratified", random_state=lf.SEED).fit(X[train], y[train]).predict_proba(X[test])[:, 1],
    "my rule baseline (w04, frozen)": lb.baseline_score(df).to_numpy()[test],
}
for name, model in models.items():
    model.fit(X[train], y[train])
    scores[name] = model.predict_proba(X[test])[:, 1]

table = pd.DataFrame({name: evaluate(s) for name, s in scores.items()}).T.round(3)
table["test base rate"] = round(y[test].mean(), 3)
display(table)

lr, base = table.loc["logistic regression"], table.loc["my rule baseline (w04, frozen)"]
print(f"logistic regression vs my rule: P@20 {lr['P@20']:.2f} vs {base['P@20']:.2f} | P@50 {lr['P@50']:.2f} vs {base['P@50']:.2f} | P@200 {lr['P@200']:.2f} vs {base['P@200']:.2f}")
best_other = table.drop(["dummy (random, stratified)", "my rule baseline (w04, frozen)", "logistic regression"])["P@50"].idxmax()
print(f"best of the complex models at P@50: {best_other} ({table.loc[best_other, 'P@50']:.2f}), it does not beat logistic regression ({lr['P@50']:.2f}), so logistic regression stays my model.")
print("random forest has the best AUC, its better at sorting the middle of the list, but the top of the list is what editors use.")
print("\nthe depth-3 tree, printed so a human can read the logic it found:")
print(export_text(models["decision tree (depth 3)"], feature_names=list(X.columns), decimals=2))

,P@20,P@50,P@100,P@200,AUC,avg precision,test base rate
"dummy (random, stratified)",0.25,0.32,0.39,0.405,0.505,0.393,0.391
"my rule baseline (w04, frozen)",0.70,0.72,0.73,0.670,0.804,0.640,0.391
logistic regression,0.70,0.82,0.81,0.790,0.844,0.715,0.391
decision tree (depth 3),0.65,0.68,0.63,0.630,0.814,0.635,0.391
random forest,0.70,0.74,0.77,0.800,0.870,0.739,0.391
gradient boosting,0.65,0.74,0.80,0.785,0.854,0.712,0.391


logistic regression vs my rule: P@20 0.70 vs 0.70 | P@50 0.82 vs 0.72 | P@200 0.79 vs 0.67
best of the complex models at P@50: random forest (0.74), it does not beat logistic regression (0.82), so logistic regression stays my model.
random forest has the best AUC, its better at sorting the middle of the list, but the top of the list is what editors use.

the depth-3 tree, printed so a human can read the logic it found:
|--- prev30_empty <= 0.50
|   |--- age_at_prediction <= 282.50
|   |   |--- log_clicks_prev30 <= 2.25
|   |   |   |--- class: 1
|   |   |--- log_clicks_prev30 >  2.25
|   |   |   |--- class: 1
|   |--- age_at_prediction >  282.50
|   |   |--- age_at_prediction <= 330.00
|   |   |   |--- class: 1
|   |   |--- age_at_prediction >  330.00
|   |   |   |--- class: 0
|--- prev30_empty >  0.50
|   |--- class: 0



## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
"""Reading the errors before believing the score. What the model leans on, where it goes wrong, and 3 concrete wrong cases.
trend_pct is used below ONLY to explain mistakes after the fact, its the label source and never a feature."""

from sklearn.inspection import permutation_importance

lr_model = models["logistic regression"]
coefs = pd.Series(lr_model[-1].coef_[0], index=X.columns)
perm = permutation_importance(lr_model, X[test], y[test], scoring="average_precision", n_repeats=10, random_state=lf.SEED)
lean = pd.DataFrame({"weight (standardised)": coefs, "permutation importance (test AP drop)": perm.importances_mean})
lean = lean.reindex(lean["permutation importance (test AP drop)"].sort_values(ascending=False).index)
display(lean.head(8).round(4))

t = df.loc[test, ["client_id", "content_type", "impressions_prev_30d", "impr_d61_90", "content_age_days", "ctr_prev30", "trend_direction", "trend_pct", "is_declining_label"]].copy()
t["score"] = scores["logistic regression"]
t["momentum_pct"] = (t["impressions_prev_30d"] / t["impr_d61_90"].clip(lower=1) - 1) * 100
t["rank"] = t["score"].rank(ascending=False, method="first").astype(int)
top200 = t[t["rank"] <= 200]

# where is it wrong
client_share = top200["client_id"].value_counts(normalize=True)
print(f"1) concentration: {client_share.iloc[0]:.0%} of the model's top 200 test picks come from ONE client. so P@50 here mostly measures how well it ranks inside that client.")
within = {f"test client {i}": roc_auc_score(g["is_declining_label"], g["score"])
          for i, (_, g) in enumerate(t.groupby("client_id"), start=1) if g["is_declining_label"].nunique() == 2 and len(g) >= 30}
print(f"   inside each test client (30+ rows) the AUC is {min(within.values()):.2f}-{max(within.values()):.2f}, so it does rank pages inside a client, not just pick the right client.")
fp = top200[top200["is_declining_label"] == 0]
near = fp["trend_pct"].between(-20, -10, inclusive="right").mean()
print(f"2) near misses: {len(fp)} of the top 200 did not decline, and {near:.0%} of those dropped 10-20%, just short of the -20% line. the label is a hard cut, the model ranks 'losing impressions' and some of its 'mistakes' lost them too, just not enough.")
tiny = t[(t["is_declining_label"] == 1) & (t["impressions_prev_30d"] <= 5)]
print(f"3) label noise at the bottom: {len(tiny)} test pages with 5 or fewer impressions in days 31-60 count as 'declined' (e.g. 1 impression -> 0 = -100%). the model ranks them low, which I think is right even though it counts as a miss.")
b_top = set(np.argsort(-scores["my rule baseline (w04, frozen)"], kind="stable")[:50])
m_top = set(np.argsort(-scores["logistic regression"], kind="stable")[:50])
print(f"4) model and rule pick different pages: only {len(b_top & m_top)} of their top 50 overlap, the rule likes big sliding pages, the model likes young pages with impressions but few clicks.")

print("\n3 concrete wrong cases (top-ranked test pages that did NOT decline):")
cases = fp.sort_values("rank").head(3)
display(cases[["rank", "score", "content_type", "impressions_prev_30d", "impr_d61_90", "momentum_pct", "content_age_days", "ctr_prev30", "trend_direction", "trend_pct"]].round(2))
for _, r in cases.iterrows():
    why = []
    if -20 < r["trend_pct"] <= -10:
        why.append(f"it actually fell {r['trend_pct']:.1f}%, a near miss on the -20% line")
    if r["ctr_prev30"] == 0:
        why.append("impressions but zero clicks, the exact profile the model has learned to fear")
    if r["content_age_days"] < 180:
        why.append(f"young page ({r['content_age_days']} days), young pages decline more on average but not all of them")
    if r["impressions_prev_30d"] < 100:
        why.append("small numbers, a few impressions either way flips the label")
    print(f"- rank {r['rank']}: " + "; ".join(why))

print("""
top 3 features and why they make sense:
- prev30_empty (strong negative): no impressions in days 31-60 means nothing to lose, the label cant say 'down'. its structural, known at day -30, not a leak (w03 test).
- age_at_prediction (younger = more risk): the same OPPOSITE result as the w04 signal audit, newer pages are still settling and fall back more often.
- log_impr_prev30 up, log_clicks_prev30 down: lots of impressions without clicks = risk, pages that earn clicks hold. plausibly Google showing a page it then stops showing, but thats my reading, the data only shows the pattern.
nothing towers like a leak would. the biggest permutation drop is prev30_empty, and thats the structural floor of the label (known at day -30), not a peek at the answer. a real leak looks like w03: AUC jumping to 0.97-1.00. seeds are fixed at 42, rerunning gives the same table. tree models can move a few points between scikit-learn versions, logistic regression barely moves.""")

,weight (standardised),permutation importance (test AP drop)
prev30_empty,-2.4553,0.1313
age_at_prediction,-0.3629,0.0986
log_impr_prev30,0.2280,0.0647
log_clicks_prev30,-0.3382,0.0599
log_impr_d61_90,0.2144,0.0453
has_keyword_data,0.0434,0.0367
content_type_keyword article,0.0379,0.0231
log_clicks_d61_90,-0.1235,0.0210


1) concentration: 94% of the model's top 200 test picks come from ONE client. so P@50 here mostly measures how well it ranks inside that client.
   inside each test client (30+ rows) the AUC is 0.66-0.90, so it does rank pages inside a client, not just pick the right client.
2) near misses: 42 of the top 200 did not decline, and 26% of those dropped 10-20%, just short of the -20% line. the label is a hard cut, the model ranks 'losing impressions' and some of its 'mistakes' lost them too, just not enough.
3) label noise at the bottom: 231 test pages with 5 or fewer impressions in days 31-60 count as 'declined' (e.g. 1 impression -> 0 = -100%). the model ranks them low, which I think is right even though it counts as a miss.
4) model and rule pick different pages: only 5 of their top 50 overlap, the rule likes big sliding pages, the model likes young pages with impressions but few clicks.

3 concrete wrong cases (top-ranked test pages that did NOT decline):


,rank,score,content_type,impressions_prev_30d,impr_d61_90,momentum_pct,content_age_days,ctr_prev30,trend_direction,trend_pct
10895,6,0.83,keyword article,1360,2533,-46.31,125,0.0,stable,-19.8
18309,7,0.83,keyword article,781,931,-16.11,125,0.0,stable,-17.8
25418,14,0.82,feedly article,29,42,-30.95,116,0.0,up,27.6


- rank 6: it actually fell -19.8%, a near miss on the -20% line; impressions but zero clicks, the exact profile the model has learned to fear; young page (125 days), young pages decline more on average but not all of them
- rank 7: it actually fell -17.8%, a near miss on the -20% line; impressions but zero clicks, the exact profile the model has learned to fear; young page (125 days), young pages decline more on average but not all of them
- rank 14: impressions but zero clicks, the exact profile the model has learned to fear; young page (116 days), young pages decline more on average but not all of them; small numbers, a few impressions either way flips the label

top 3 features and why they make sense:
- prev30_empty (strong negative): no impressions in days 31-60 means nothing to lose, the label cant say 'down'. its structural, known at day -30, not a leak (w03 test).
- age_at_prediction (younger = more risk): the same OPPOSITE result as the w04 signal audit, newer pages are still s

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.